# 09 · LightGBM with GFS forecasts

Can a tree model use GFS forecasts productively? Same 2024 validation pairs as notebook 08.

**Key results**
- LightGBM with GFS: 8.82 MU; without GFS: 10.67 MU — forecasts help trees, but not enough.
- Still well behind Chronos-2 calendar LoRA (6.87 MU).

> Paths are relative to `notebooks/`. Outputs are from the original research run; see [`data/README.md`](../data/README.md) for data that must be downloaded.

➡ Next: [10 · Temporal Convolutional Network and the ensemble](10_tcn_and_ensemble.ipynb)

Imports

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA = Path("../data/processed")

print("LightGBM:", lgb.__version__)

LightGBM: 4.7.0


Load datasets

In [3]:
demand = pd.read_csv(
    DATA / "gridpulse_v2_development.csv",
    parse_dates=["date"]
).sort_values("date")

gfs = pd.read_parquet(
    DATA / "gridpulse_gfs_previous_runs_2021_2024.parquet"
)

gfs["time"] = pd.to_datetime(gfs["time"])

benchmark = pd.read_parquet(
    DATA / "gridpulse_chronos2_validation_2024.parquet"
)

benchmark["origin"] = pd.to_datetime(benchmark["origin"])
benchmark["target_date"] = pd.to_datetime(benchmark["target_date"])

print("Demand:", demand.shape)
print("GFS hourly:", gfs.shape)
print("Benchmark:", benchmark.shape)

assert benchmark["origin"].nunique() == 360
assert len(benchmark) == 2520

Demand: (2193, 4)
GFS hourly: (32904, 8)
Benchmark: (2520, 7)


Aggregate hourly forecasts

In [4]:
lead_cols = [
    f"temperature_2m_previous_day{i}"
    for i in range(1, 8)
]

gfs_daily = (
    gfs.assign(date=gfs["time"].dt.normalize())
    .groupby("date")[lead_cols]
    .agg(["mean", "max", "min", "count"])
)

gfs_daily.columns = [
    f"{lead}_{stat}"
    for lead, stat in gfs_daily.columns
]

# Reject incomplete hourly forecasts
for lead in lead_cols:
    incomplete = gfs_daily[f"{lead}_count"] != 24

    for stat in ["mean", "max", "min"]:
        gfs_daily.loc[incomplete, f"{lead}_{stat}"] = np.nan

print("Daily GFS records:", len(gfs_daily))
display(gfs_daily.head(3))

Daily GFS records: 1371


,temperature_2m_previous_day1_mean,temperature_2m_previous_day1_max,temperature_2m_previous_day1_min,temperature_2m_previous_day1_count,temperature_2m_previous_day2_mean,temperature_2m_previous_day2_max,temperature_2m_previous_day2_min,temperature_2m_previous_day2_count,temperature_2m_previous_day3_mean,temperature_2m_previous_day3_max,...,temperature_2m_previous_day5_min,temperature_2m_previous_day5_count,temperature_2m_previous_day6_mean,temperature_2m_previous_day6_max,temperature_2m_previous_day6_min,temperature_2m_previous_day6_count,temperature_2m_previous_day7_mean,temperature_2m_previous_day7_max,temperature_2m_previous_day7_min,temperature_2m_previous_day7_count
date,,,,,,,,,,,,,,,,,,,,,
2021-04-01,29.841667,36.0,23.5,24,30.191667,36.5,24.1,24,29.933333,36.2,...,23.9,24,29.837500,36.5,23.5,24,29.570833,35.2,24.2,24
2021-04-02,29.812500,36.2,23.8,24,29.891667,36.3,23.9,24,29.520833,35.8,...,24.6,24,30.820833,37.5,24.5,24,30.954167,37.6,23.7,24
2021-04-03,30.341667,36.9,23.5,24,30.758333,37.0,23.7,24,30.841667,37.2,...,24.6,24,31.208333,37.6,24.8,24,32.037500,38.8,25.4,24


Create the origin–target weather lookup

In [5]:
weather_rows = []

for origin in pd.date_range("2021-04-01", "2024-12-24"):
    for h in range(1, 8):
        target_date = origin + pd.Timedelta(days=h)
        lead = f"temperature_2m_previous_day{h}"

        if target_date not in gfs_daily.index:
            continue

        row = gfs_daily.loc[target_date]

        weather_rows.append({
            "origin": origin,
            "target_date": target_date,
            "horizon": h,
            "gfs_temp_mean": row[f"{lead}_mean"],
            "gfs_temp_max": row[f"{lead}_max"],
            "gfs_temp_min": row[f"{lead}_min"],
        })

weather_lookup = pd.DataFrame(weather_rows)

print("Forecast records:", len(weather_lookup))
print("Missing forecasts:",
      weather_lookup["gfs_temp_mean"].isna().sum())

display(weather_lookup.head(7))

Forecast records: 9548
Missing forecasts: 168


,origin,target_date,horizon,gfs_temp_mean,gfs_temp_max,gfs_temp_min
0,2021-04-01,2021-04-02,1,29.812500,36.2,23.8
1,2021-04-01,2021-04-03,2,30.758333,37.0,23.7
2,2021-04-01,2021-04-04,3,32.220833,38.8,25.6
3,2021-04-01,2021-04-05,4,35.075000,42.2,27.9
4,2021-04-01,2021-04-06,5,35.712500,42.0,30.7
5,2021-04-01,2021-04-07,6,34.104167,39.5,29.5
6,2021-04-01,2021-04-08,7,30.979167,36.3,26.0


In [6]:
demand_series = demand.set_index("date")["energy_met_mu"].sort_index()

origin_features = []

for origin in pd.date_range("2021-04-01", "2024-12-24"):
    history = demand_series.reindex(
        pd.date_range(end=origin, periods=56, freq="D")
    )

    # Require 56 valid historical demand observations
    if history.isna().any():
        continue

    origin_features.append({
        "origin": origin,
        "demand_lag_1": history.iloc[-1],
        "demand_lag_7": history.iloc[-7],
        "demand_lag_14": history.iloc[-14],
        "demand_lag_28": history.iloc[-28],
        "demand_mean_7": history.iloc[-7:].mean(),
        "demand_mean_14": history.iloc[-14:].mean(),
        "demand_mean_28": history.iloc[-28:].mean(),
        "demand_std_14": history.iloc[-14:].std(),
        "demand_std_28": history.iloc[-28:].std(),
        "demand_trend": (
            history.iloc[-7:].mean()
            - history.iloc[-28:-7].mean()
        ),
    })

origin_df = pd.DataFrame(origin_features)

print("Valid demand origins:", len(origin_df))
display(origin_df.head(3))

Valid demand origins: 1034


,origin,demand_lag_1,demand_lag_7,demand_lag_14,demand_lag_28,demand_mean_7,demand_mean_14,demand_mean_28,demand_std_14,demand_std_28,demand_trend
0,2021-04-24,59.7,67.6,75.5,65.0,67.057143,74.378571,73.621429,10.295270,8.715689,-8.752381
1,2021-04-25,61.1,79.9,83.6,56.3,66.128571,73.350000,73.482143,10.877482,8.888030,-9.804762
2,2021-04-26,66.3,69.5,85.8,70.2,64.185714,72.114286,73.839286,10.602674,8.357092,-12.871429


In [7]:
dataset = weather_lookup.merge(
    origin_df,
    on="origin",
    how="inner",
    validate="many_to_one"
)

# Target-date calendar features
target = dataset["target_date"]

dataset["day_of_week"] = target.dt.dayofweek
dataset["month"] = target.dt.month
dataset["day_of_year"] = target.dt.dayofyear
dataset["is_weekend"] = (target.dt.dayofweek >= 5).astype(int)

dataset["year_sin"] = np.sin(
    2 * np.pi * target.dt.dayofyear / 365.25
)
dataset["year_cos"] = np.cos(
    2 * np.pi * target.dt.dayofyear / 365.25
)

# Actual target values are labels, never input features
dataset = dataset.merge(
    demand[["date", "energy_met_mu"]].rename(
        columns={
            "date": "target_date",
            "energy_met_mu": "actual_mu"
        }
    ),
    on="target_date",
    how="left",
    validate="many_to_one"
)

print("Dataset rows:", len(dataset))
print("Missing demand targets:", dataset["actual_mu"].isna().sum())
print("Missing GFS forecasts:", dataset["gfs_temp_mean"].isna().sum())

display(dataset.head(3))

Dataset rows: 7238
Missing demand targets: 34
Missing GFS forecasts: 168


,origin,target_date,horizon,gfs_temp_mean,gfs_temp_max,gfs_temp_min,demand_lag_1,demand_lag_7,demand_lag_14,demand_lag_28,...,demand_std_14,demand_std_28,demand_trend,day_of_week,month,day_of_year,is_weekend,year_sin,year_cos,actual_mu
0,2021-04-24,2021-04-25,1,32.854167,38.6,26.7,59.7,67.6,75.5,65.0,...,10.29527,8.715689,-8.752381,6,4,115,1,0.918121,-0.396299,61.1
1,2021-04-24,2021-04-26,2,35.116667,40.8,28.8,59.7,67.6,75.5,65.0,...,10.29527,8.715689,-8.752381,0,4,116,0,0.911169,-0.412034,66.3
2,2021-04-24,2021-04-27,3,37.337500,42.9,31.6,59.7,67.6,75.5,65.0,...,10.29527,8.715689,-8.752381,1,4,117,0,0.903946,-0.427646,70.8


In [8]:
features = [
    "horizon",
    "gfs_temp_mean", "gfs_temp_max", "gfs_temp_min",
    "demand_lag_1", "demand_lag_7",
    "demand_lag_14", "demand_lag_28",
    "demand_mean_7", "demand_mean_14", "demand_mean_28",
    "demand_std_14", "demand_std_28",
    "demand_trend",
    "day_of_week", "month", "day_of_year",
    "is_weekend", "year_sin", "year_cos",
]

train = dataset.loc[
    (dataset["target_date"] < "2024-01-01")
    & dataset["actual_mu"].notna()
].copy()

# Use the exact frozen benchmark origins and target dates.
validation = benchmark[
    ["origin", "target_date", "horizon", "actual_mu",
     "chronos2_lora_calendar_200"]
].merge(
    dataset.drop(columns="actual_mu"),
    on=["origin", "target_date", "horizon"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Training rows:", len(train))
print("Validation rows:", len(validation))
print("Unmatched validation rows:",
      (validation["_merge"] != "both").sum())

assert train["target_date"].max() < pd.Timestamp("2024-01-01")
assert len(validation) == 2520
assert (validation["_merge"] == "both").all()

Training rows: 4663
Validation rows: 2520
Unmatched validation rows: 0


In [9]:
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=5,
    min_child_samples=30,
    reg_lambda=2.0,
    random_state=42,
    verbosity=-1,
    n_jobs=-1,
)

model.fit(
    train[features],
    train["actual_mu"]
)

validation["lightgbm_pred"] = model.predict(
    validation[features]
)

for name in [
    "chronos2_lora_calendar_200",
    "lightgbm_pred",
]:
    mae = mean_absolute_error(
        validation["actual_mu"],
        validation[name]
    )

    print(f"{name:30s} MAE = {mae:.4f} MU")

chronos2_lora_calendar_200     MAE = 6.8693 MU
lightgbm_pred                  MAE = 8.8214 MU


In [10]:
no_gfs_features = [
    col for col in features
    if not col.startswith("gfs_")
]

control = lgb.LGBMRegressor(
    **model.get_params()
)

control.fit(
    train[no_gfs_features],
    train["actual_mu"]
)

validation["lightgbm_no_gfs"] = control.predict(
    validation[no_gfs_features]
)

for name in [
    "chronos2_lora_calendar_200",
    "lightgbm_pred",
    "lightgbm_no_gfs",
]:
    mae = mean_absolute_error(
        validation["actual_mu"],
        validation[name]
    )
    print(f"{name:30s} MAE = {mae:.4f} MU")

chronos2_lora_calendar_200     MAE = 6.8693 MU
lightgbm_pred                  MAE = 8.8214 MU
lightgbm_no_gfs                MAE = 10.6702 MU
